# LabMiM - Treinamento de Modelos no Google Colab

Este notebook foi configurado para executar o script de treinamento de alta performance do LabMiM no ambiente do Google Colab usando as placas de vídeo gratuitas (Nvidia T4).

**Aviso:** Certifique-se de que a Aceleração de Hardware está ativada. Vá em `Runtime > Change runtime type` e escolha **T4 GPU**.

In [ ]:
from google.colab import drive

# 1. Monta o Google Drive
drive.mount("/content/drive")

# 2. Clona o repositório
!git clone https://github.com/Bruno-Mascarenhas/micrometeorology-ufba.git
%cd micrometeorology-ufba

# 3. Instala as dependências mantendo o PyTorch da GPU
!pip install -e ".[tcc]"

## 1. Treinamento Base: LSTM (Baseline)

Abaixo o comando para treinar um LSTM padrão para correção de radiação solar.

In [ ]:
# Inicia o TensorBoard em background
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/LabMiM/logs

# Treina o LSTM
!python scripts/train_colab.py \
    --train_path /content/drive/MyDrive/LabMiM/dados_treino.csv \
    --val_path /content/drive/MyDrive/LabMiM/dados_val.csv \
    --out_model /content/drive/MyDrive/LabMiM/modelos/lstm_base.pt \
    --log_dir /content/drive/MyDrive/LabMiM/logs/lstm_base \
    --model_type lstm \
    --batch_size 1024 \
    --lr 0.001 \
    --epochs 150 \
    --features radiacao_obs temp pressao umidade \
    --target radiacao_real

## 2. Experimento Avançado: Transformer

O Transformer é muito mais sensível aos hiperparâmetros, mas pode superar o LSTM se o tamanho do dataset for enorme. Sugestão: diminua a `lr` e aumente a paciência.

In [ ]:
!python scripts/train_colab.py \
    --train_path /content/drive/MyDrive/LabMiM/dados_treino.csv \
    --val_path /content/drive/MyDrive/LabMiM/dados_val.csv \
    --out_model /content/drive/MyDrive/LabMiM/modelos/transformer_base.pt \
    --log_dir /content/drive/MyDrive/LabMiM/logs/transformer_base \
    --model_type transformer \
    --hidden_size 128 \
    --layers 3 \
    --batch_size 512 \
    --lr 0.0005 \
    --patience 30 \
    --epochs 200 \
    --features radiacao_obs temp pressao umidade \
    --target radiacao_real

## 3. Avaliação e Comparação Métrica

Use esta célula abaixo para carregar os pesos que você salvou no drive e comparar as predições do modelo com os dados de validação.

In [ ]:
from solrad_correction.models.lstm import LSTMRegressor

# Exemplo rápido de inferência após o treino
# 1. Carrega o modelo treinado
model = LSTMRegressor(input_size=4, hidden_size=64, num_layers=2)
model.load("/content/drive/MyDrive/LabMiM/modelos/lstm_base.pt")

# 2. Faz as predições (adapte o load_data ou use o predict direto num dataset)
# preds = model.predict(seu_dataset_de_teste)
# plt.plot(preds)
# plt.plot(ground_truth)
# plt.show()